<a href="https://colab.research.google.com/github/aliakseizvertouski/olist/blob/main/olist_draft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
customers = pd.read_csv('/content/olist/olist_customers_dataset.csv')
geo = pd.read_csv('/content/olist/olist_geolocation_dataset.csv')
order_items = pd.read_csv('/content/olist/olist_order_items_dataset.csv')
order_payments = pd.read_csv('/content/olist/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('/content/olist/olist_order_reviews_dataset.csv')
orders = pd.read_csv('/content/olist/olist_orders_dataset.csv')
products = pd.read_csv('/content/olist/olist_products_dataset.csv')
sellers = pd.read_csv('/content/olist/olist_sellers_dataset.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/content/olist/olist_customers_dataset.csv'

# Информация о датасете

In [ ]:
print (customers.columns)
print (geo.columns)
print (order_items.columns)
print (order_payments.columns)
print (order_reviews.columns)
print (orders.columns)
print (products.columns)
print (sellers.columns)

# Соблюдение сроков доставки

In [ ]:
delivered_orders = orders[orders['order_status'] == 'delivered'].copy()

delivered_orders['estimated_date'] = pd.to_datetime(delivered_orders['order_estimated_delivery_date'])
delivered_orders['actual_date'] = pd.to_datetime(delivered_orders['order_delivered_customer_date'])


# Если дата доставки больше (позже) обещанной, там будет True, иначе False
delivered_orders['is_late'] = delivered_orders['actual_date'] > delivered_orders['estimated_date']

# 4. Считаем количество доставок вовремя и с опозданием
# False - вовремя, True - с опозданием
delivery_stats = delivered_orders['is_late'].value_counts()
delivery_stats.index = ['вовремя', 'опоздание'] # Переименуем для понятности

total_orders = len(delivered_orders)
late_percent = (delivery_stats['опоздание'] / total_orders) * 100

print(f"Всего доставлено заказов: {total_orders}")
print(delivery_stats)
print(f"Процент опозданий: {late_percent:.2f}%\n")

In [ ]:
plt.figure(figsize=(7, 7))

plt.pie(
    delivery_stats,
    labels=delivery_stats.index,
    autopct='%1.1f%%',
    colors=['#4CAF50', '#F44336'],
    startangle=90
)

plt.title('Соблюдение сроков доставки заказов')
plt.show()

## Самый долгий этап заказа

In [ ]:
df = orders[orders['order_status'] == 'delivered'].copy()

# Список колонок, которые нужно превратить в даты
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date'
]

# Превращаем строки в даты
for col in date_columns:
    df[col] = pd.to_datetime(df[col])

# 2. Считаем время между этапами в днях
# Используем .dt.total_seconds() / 86400, чтобы получить дробное число дней (например, 1.5 дня)
df['days_to_approve'] = (df['order_approved_at'] - df['order_purchase_timestamp']).dt.total_seconds() / 86400
df['days_to_carrier'] = (df['order_delivered_carrier_date'] - df['order_approved_at']).dt.total_seconds() / 86400
df['days_to_customer'] = (df['order_delivered_customer_date'] - df['order_delivered_carrier_date']).dt.total_seconds() / 86400

# 3. Считаем среднее время для каждого этапа
mean_times = {
    '1. Подтверждение': df['days_to_approve'].mean(),
    '2. Сборка и склад': df['days_to_carrier'].mean(),
    '3. Доставка курьером': df['days_to_customer'].mean()
}

# Превращаем в Series для удобства построения графика
results = pd.Series(mean_times)

print("Среднее время этапов (в днях):")
print(results.round(2))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
results.plot(kind='bar', color='skyblue')

plt.title('Среднее время прохождения этапов заказа')
plt.ylabel('Дни')
plt.xlabel('Этапы')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


delivered_orders = orders[orders['order_status'] == 'delivered'].copy()

df_geo = delivered_orders.merge(customers, on='customer_id')

df_geo['purchase'] = pd.to_datetime(df_geo['order_purchase_timestamp'])
df_geo['actual_delivery'] = pd.to_datetime(df_geo['order_delivered_customer_date'])
df_geo['estimated_delivery'] = pd.to_datetime(df_geo['order_estimated_delivery_date'])

df_geo['actual_days'] = (df_geo['actual_delivery'] - df_geo['purchase']).dt.total_seconds() / 86400
df_geo['estimated_days'] = (df_geo['estimated_delivery'] - df_geo['purchase']).dt.total_seconds() / 86400

state_stats = df_geo.groupby('customer_state')[['actual_days', 'estimated_days']].mean()

top_10_slowest = state_stats.sort_values(by='actual_days', ascending=False).head(10)

print("В какие штатты дольше всего доставляется товар?")
print(top_10_slowest.round(1))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
top_10_slowest.plot(kind='bar', ax=ax, color=['#f44336', '#2196f3'])

plt.title('Топ-10 самых "медленных" штатов: Факт vs Обещание')
plt.xlabel('Штат')
plt.ylabel('Дни')
plt.legend(['Реальный срок (дни)', 'Обещанный срок (дни)'])
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()